# KL-Perturbed Edge-DP: Multi-Dataset Benchmark

Consolidated notebook that runs the MM-edgeDP dictionary-retrieval method
and its three baselines (Gaussian SGC, Edge RR, GAP-EDP) on four datasets
from a single config switch:

- `cora` (Planetoid)
- `citeseer` (Planetoid)
- `pubmed` (Planetoid)
- `ogbn-arxiv` (OGB)

Structure:
1. Environment + imports
2. Dataset loader + stratified split (small and large graphs handled uniformly)
3. MM-edgeDP pipeline (dictionary build → exponential-mechanism assignment → GNN training)
4. Baseline 1: Gaussian SGC (edge-DP, continuous perturbation)
5. Baseline 2: Edge Randomized Response (edge-DP, adjacency perturbation)
6. Baseline 3: GAP-EDP (aggregation perturbation; note: node-DP guarantee)
7. Edge-inference attacks (Threat Model A black-box, Threat Model B white-box)

Change `DATASET_NAME` in Section 2 to pick a dataset and re-run the notebook.


## 1. Environment setup


In [ ]:
# ==========================================
# CELL 1: ENVIRONMENT & IMPORTS
# ==========================================
import importlib.util
import subprocess
import sys


def ensure_package(module_name, pip_name=None):
    if importlib.util.find_spec(module_name) is None:
        pkg = pip_name or module_name
        print(f'Installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])


ensure_package('torch_geometric', 'torch-geometric')
ensure_package('ogb', 'ogb')
ensure_package('sklearn', 'scikit-learn')

import os
import math
import time
import random
import pickle
from contextlib import contextmanager

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.utils import (
    subgraph, k_hop_subgraph, to_undirected, coalesce, dropout_edge,
)
from torch_geometric.nn import GCNConv, global_mean_pool

from ogb.nodeproppred import PygNodePropPredDataset

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

try:
    import pandas as pd
except Exception:
    pd = None

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'PyTorch version: {torch.__version__}')
print(f'Active device: {device}')


## 2. Dataset selection and loader

Flip `DATASET_NAME` to one of `cora`, `citeseer`, `pubmed`, `ogbn-arxiv`.

The dataset presets control everything that needs to scale with graph size:
- `max_private_nodes` / `max_val_nodes`: cap the number of private queries and val
  subgraphs so ogbn-arxiv fits in a reasonable wall-clock budget.
- `dict_per_class`: bigger classes (arxiv has 40) benefit from smaller per-class
  dictionaries; small graphs with 3–7 classes can afford more.
- `public_frac`: Planetoid datasets use a larger public share because they are small;
  ogbn-arxiv uses 2% as in the paper.


In [ ]:
# ==========================================
# CELL 2: DATASET SELECTION + LOADING UTILITIES
# ==========================================

# -------- Change this to switch datasets --------
DATASET_NAME = 'cora'   # 'cora' | 'citeseer' | 'pubmed' | 'ogbn-arxiv'
SEED = 42
# ------------------------------------------------


DATASET_PRESETS = {
    'cora': {
        'public_frac': 0.20,
        'val_frac':    0.20,
        'max_private_nodes': None,
        'max_val_nodes':     None,
        'dict_per_class':    50,
        'walk_hops': 2,
        'query_hops': 1,
        'epochs': 50,
        'batch_size': 32,
        'hidden_channels': 16,
    },
    'citeseer': {
        'public_frac': 0.20,
        'val_frac':    0.20,
        'max_private_nodes': None,
        'max_val_nodes':     None,
        'dict_per_class':    50,
        'walk_hops': 2,
        'query_hops': 1,
        'epochs': 50,
        'batch_size': 32,
        'hidden_channels': 16,
    },
    'pubmed': {
        'public_frac': 0.10,
        'val_frac':    0.10,
        'max_private_nodes': 10000,
        'max_val_nodes':     2000,
        'dict_per_class':    50,
        'walk_hops': 2,
        'query_hops': 1,
        'epochs': 25,
        'batch_size': 64,
        'hidden_channels': 32,
    },
    'ogbn-arxiv': {
        'public_frac': 0.02,
        'val_frac':    0.02,
        'max_private_nodes': 30000,
        'max_val_nodes':     3000,
        'dict_per_class':    48,
        'walk_hops': 1,
        'query_hops': 2,
        'epochs': 8,
        'batch_size': 64,
        'hidden_channels': 32,
    },
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _register_pyg_safe_globals():
    """Allowlist PyG classes for PyTorch 2.6+ weights-only unpickling (needed for OGB)."""
    if not hasattr(torch.serialization, 'add_safe_globals'):
        return
    safe = []
    try:
        from torch_geometric.data.data import Data as _Data
        safe.append(_Data)
    except Exception:
        pass
    try:
        from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
        safe.extend([DataEdgeAttr, DataTensorAttr])
    except Exception:
        pass
    try:
        from torch_geometric.data.storage import (
            BaseStorage, NodeStorage, EdgeStorage, GlobalStorage,
        )
        safe.extend([BaseStorage, NodeStorage, EdgeStorage, GlobalStorage])
    except Exception:
        pass
    if len(safe) > 0:
        torch.serialization.add_safe_globals(safe)


@contextmanager
def _torch_load_weights_only_false():
    original_torch_load = torch.load

    def patched(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return original_torch_load(*args, **kwargs)

    torch.load = patched
    try:
        yield
    finally:
        torch.load = original_torch_load


def load_graph_dataset(dataset_name):
    name = dataset_name.lower()

    if name == 'ogbn-arxiv':
        _register_pyg_safe_globals()
        try:
            dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='data/ogb')
        except pickle.UnpicklingError:
            with _torch_load_weights_only_false():
                dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='data/ogb')
        data = dataset[0]
        data.y = data.y.squeeze(-1).long()
        data.x = data.x.float()
        num_classes = int(dataset.num_classes)

    elif name in ('cora', 'citeseer', 'pubmed'):
        pretty = name.capitalize() if name != 'citeseer' else 'CiteSeer'
        pretty = 'PubMed' if name == 'pubmed' else pretty
        dataset = Planetoid(root=f'data/Planetoid/{pretty}', name=pretty,
                            transform=T.NormalizeFeatures())
        data = dataset[0]
        data.y = data.y.long()
        data.x = data.x.float()
        num_classes = int(dataset.num_classes)
    else:
        raise ValueError(f'Unsupported dataset: {dataset_name}')

    data.edge_index = to_undirected(data.edge_index, num_nodes=data.num_nodes)
    data.edge_index = coalesce(data.edge_index, num_nodes=data.num_nodes)

    # L2-normalise so the Edge-DP sensitivity analysis (|x_v|_2 <= 1) holds.
    data.x = F.normalize(data.x, p=2, dim=1)
    return dataset, data, num_classes


def stratified_split_indices(y, public_frac=0.02, val_frac=0.02, seed=0):
    g = torch.Generator().manual_seed(seed)
    classes = torch.unique(y)
    pub, val, train = [], [], []
    for c in classes.tolist():
        idx = torch.where(y == c)[0]
        if idx.numel() == 0:
            continue
        perm = idx[torch.randperm(idx.numel(), generator=g)]
        n_pub = max(1, int(public_frac * idx.numel()))
        n_val = max(1, int(val_frac * idx.numel()))
        n_pub = min(n_pub, idx.numel())
        n_val = min(n_val, max(idx.numel() - n_pub, 0))
        pub.append(perm[:n_pub])
        val.append(perm[n_pub:n_pub + n_val])
        train.append(perm[n_pub + n_val:])

    public_nodes = torch.cat(pub) if pub else torch.tensor([], dtype=torch.long)
    val_nodes = torch.cat(val) if val else torch.tensor([], dtype=torch.long)
    train_nodes = torch.cat(train) if train else torch.tensor([], dtype=torch.long)

    public_nodes = public_nodes[torch.randperm(public_nodes.numel(), generator=g)]
    val_nodes    = val_nodes[torch.randperm(val_nodes.numel(), generator=g)]
    train_nodes  = train_nodes[torch.randperm(train_nodes.numel(), generator=g)]
    return public_nodes, train_nodes, val_nodes


def cap_indices(indices, max_count=None, seed=0):
    if max_count is None or indices.numel() <= max_count:
        return indices
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(indices.numel(), generator=g)
    return indices[perm[:max_count]]


def split_edges_by_nodes(edge_index, public_nodes, train_nodes, val_nodes, num_nodes):
    pub_ei, _     = subgraph(public_nodes, edge_index, relabel_nodes=False, num_nodes=num_nodes)
    private_ei, _ = subgraph(train_nodes,  edge_index, relabel_nodes=False, num_nodes=num_nodes)
    val_ei, _     = subgraph(val_nodes,    edge_index, relabel_nodes=False, num_nodes=num_nodes)
    return pub_ei, private_ei, val_ei


# -------- Load dataset + build splits --------
set_seed(SEED)
if DATASET_NAME not in DATASET_PRESETS:
    raise ValueError(f'Unknown DATASET_NAME={DATASET_NAME}. Options: {list(DATASET_PRESETS.keys())}')
PRESET = DATASET_PRESETS[DATASET_NAME]
print(f'Loading dataset: {DATASET_NAME}')
dataset, data, num_classes = load_graph_dataset(DATASET_NAME)
x_l2 = data.x  # already L2-normalised in load_graph_dataset

print(f'Dataset={DATASET_NAME} | Nodes={data.num_nodes} | Edges={data.edge_index.size(1)} | Classes={num_classes} | FeatureDim={data.x.size(1)}')

public_nodes, train_nodes, val_nodes = stratified_split_indices(
    data.y,
    public_frac=PRESET['public_frac'],
    val_frac=PRESET['val_frac'],
    seed=SEED,
)
train_nodes = cap_indices(train_nodes, PRESET['max_private_nodes'], seed=SEED + 1)
val_nodes   = cap_indices(val_nodes,   PRESET['max_val_nodes'],     seed=SEED + 2)

pub_edge_index, private_edge_index, val_edge_index = split_edges_by_nodes(
    data.edge_index, public_nodes, train_nodes, val_nodes, data.num_nodes,
)

print(f'Split sizes -> Public: {len(public_nodes)} | Private train: {len(train_nodes)} | Validation: {len(val_nodes)}')
print(f'Edge splits -> Public: {pub_edge_index.size(1)} | Private: {private_edge_index.size(1)} | Val: {val_edge_index.size(1)}')

max_norm = float(torch.linalg.norm(x_l2, ord=2, dim=1).max().item())
print(f'Max node feature L2 norm after normalization: {max_norm:.6f}')


## 3. MM-edgeDP: dictionary retrieval with the exponential mechanism

This is the method described in the project PDF:

1. Build a class-conditional dictionary of public subgraphs (prototypes).
2. For each private node, compute a 1-hop mean-aggregated query.
3. Sample a prototype via the exponential mechanism on `−||q_u − f_k||_2` scores.
4. Train a GCN on the synthesised (prototype, label) dataset with feature jitter and edge dropout.


In [ ]:
# ==========================================
# CELL 3: MM-edgeDP CORE FUNCTIONS
# (dictionary build + exponential-mechanism sampling + GCN training)
# ==========================================
from collections import Counter


def one_hop_mean_aggregate(edge_index, x_features, num_nodes):
    """H_u = mean_{v in N(u)} x_v on an arbitrary edge set."""
    ei = to_undirected(edge_index, num_nodes=num_nodes)
    ei = coalesce(ei, num_nodes=num_nodes)

    H_sum = torch.zeros_like(x_features)
    if ei.numel() == 0:
        return H_sum

    row, col = ei
    H_sum.scatter_add_(0, row.unsqueeze(1).expand(-1, x_features.size(1)), x_features[col])

    counts = torch.zeros((num_nodes,), dtype=x_features.dtype, device=x_features.device)
    counts.scatter_add_(0, row, torch.ones_like(row, dtype=x_features.dtype))

    H_mean = torch.zeros_like(H_sum)
    nz = counts > 0
    H_mean[nz] = H_sum[nz] / counts[nz].unsqueeze(1)
    return H_mean


def degree_weighted_sgc_embedding(x_sub, edge_index_sub):
    n = x_sub.size(0)
    if n == 0:
        return torch.zeros(x_sub.size(1), device=x_sub.device)
    if edge_index_sub.numel() == 0:
        weights = torch.ones(n, dtype=x_sub.dtype, device=x_sub.device)
    else:
        row, col = edge_index_sub
        deg = torch.bincount(row, minlength=n).to(x_sub.dtype)
        deg = deg + torch.bincount(col, minlength=n).to(x_sub.dtype)
        weights = deg + 1.0
    ws = (x_sub * weights.unsqueeze(1)).sum(dim=0)
    return ws / weights.sum().clamp_min(1e-12)


def build_prototype_feature(edge_index_sub, x_sub, query_mode='one_hop'):
    n = x_sub.size(0)
    H1 = one_hop_mean_aggregate(edge_index_sub, x_sub, num_nodes=n)
    z1 = degree_weighted_sgc_embedding(H1, edge_index_sub)
    if query_mode == 'one_hop':
        return F.normalize(z1, p=2, dim=0)
    if query_mode == 'two_hop_concat':
        H2 = one_hop_mean_aggregate(edge_index_sub, H1, num_nodes=n)
        z2 = degree_weighted_sgc_embedding(H2, edge_index_sub)
        return F.normalize(torch.cat([z1, z2], dim=0), p=2, dim=0)
    raise ValueError(f'Unsupported query_mode: {query_mode}')


def build_private_query_features(private_edges, x_features, query_mode='one_hop'):
    n = x_features.size(0)
    H1 = one_hop_mean_aggregate(private_edges, x_features, num_nodes=n)
    if query_mode == 'one_hop':
        return F.normalize(H1, p=2, dim=1)
    if query_mode == 'two_hop_concat':
        H2 = one_hop_mean_aggregate(private_edges, H1, num_nodes=n)
        return F.normalize(torch.cat([H1, H2], dim=1), p=2, dim=1)
    raise ValueError(f'Unsupported query_mode: {query_mode}')


def build_class_conditional_public_dictionary(
    data_obj, x_features, labels_tensor, public_nodes_tensor, pub_edges,
    num_classes, dict_per_class=50, num_hops=2, query_mode='one_hop',
    min_class_fraction=1.0, max_proto_nodes=None,
):
    """Build a class-conditional dictionary of prototypes from the public split."""
    print(
        f'Building dictionary ({dict_per_class}/class, hops={num_hops}, mode={query_mode}, '
        f'max_proto_nodes={max_proto_nodes})'
    )
    if public_nodes_tensor.numel() == 0:
        raise ValueError('Public split is empty.')

    pub_u = to_undirected(pub_edges, num_nodes=data_obj.num_nodes)
    pub_u = coalesce(pub_u, num_nodes=data_obj.num_nodes)

    public_labels = labels_tensor[public_nodes_tensor]
    dictionary = []
    class_to_proto_indices = {c: [] for c in range(num_classes)}

    for c in range(num_classes):
        class_pool = public_nodes_tensor[public_labels == c]
        if class_pool.numel() == 0:
            class_pool = public_nodes_tensor

        for _ in range(dict_per_class):
            anchor = class_pool[torch.randint(0, class_pool.numel(), (1,)).item()]
            subset, _, _, _ = k_hop_subgraph(
                int(anchor.item()),
                num_hops=num_hops,
                edge_index=pub_u,
                relabel_nodes=False,
                num_nodes=data_obj.num_nodes,
            )
            proto_labels = labels_tensor[subset]
            class_mask = proto_labels == c
            class_count = int(class_mask.sum().item())
            subset_count = int(subset.numel())
            frac = class_count / max(subset_count, 1)

            if frac >= min_class_fraction and class_count > 0:
                class_subset = subset
            else:
                class_subset = subset[class_mask]
                if class_subset.numel() == 0:
                    class_subset = anchor.view(1)

            # Scale guard: cap per-prototype node count for large graphs so the
            # dictionary tensor footprint stays manageable.
            if max_proto_nodes is not None and class_subset.numel() > max_proto_nodes:
                perm = torch.randperm(class_subset.numel())[:max_proto_nodes]
                class_subset = class_subset[perm]

            sub_ei, _ = subgraph(class_subset, pub_u, relabel_nodes=True, num_nodes=data_obj.num_nodes)
            x_sub = x_features[class_subset]
            proto_feat = build_prototype_feature(sub_ei, x_sub, query_mode=query_mode)

            proto_idx = len(dictionary)
            dictionary.append({
                'edge_index': sub_ei,
                'x': x_sub,
                'proto_feat': proto_feat,
                'class': c,
            })
            class_to_proto_indices[c].append(proto_idx)

    if len(dictionary) == 0:
        raise RuntimeError('Dictionary is empty. Increase public split or relax class purity.')

    dict_feats = torch.stack([g['proto_feat'] for g in dictionary], dim=0)
    print(f'Dictionary size={len(dictionary)} | Feature dim={dict_feats.size(1)}')
    return dictionary, dict_feats, class_to_proto_indices


def gumbel_max_sample(logits):
    u = torch.rand_like(logits).clamp_(1e-8, 1 - 1e-8)
    gumbel = -torch.log(-torch.log(u))
    return torch.argmax(logits + gumbel).item()


@torch.no_grad()
def synthesize_edge_dp_assignments(
    private_edges, x_features, labels, private_nodes_tensor,
    dict_features, class_to_proto_indices, epsilon_total,
    utility_sensitivity=1.0, query_mode='one_hop', label_conditioning=True,
):
    """
    Edge-DP exponential mechanism over negative L2 distance.
    Returns assignments (proto_indices + labels) and diagnostics.
    """
    # Edge-level composition: adding/removing an edge affects at most two queries.
    epsilon_per_node = epsilon_total / 2.0

    Q = build_private_query_features(private_edges, x_features, query_mode=query_mode)
    K_global = dict_features.size(0)
    num_classes_eff = int(labels.max().item()) + 1

    all_idx = torch.arange(K_global, dtype=torch.long)
    class_idx_tensors = {}
    for c in range(num_classes_eff):
        idx_list = class_to_proto_indices.get(c, [])
        class_idx_tensors[c] = (
            torch.tensor(idx_list, dtype=torch.long) if len(idx_list) > 0 else all_idx
        )

    N = int(private_nodes_tensor.numel())
    selected_proto_indices = torch.empty(N, dtype=torch.long)
    private_labels = labels[private_nodes_tensor].long().clone()
    selection_counts = torch.zeros(K_global, dtype=torch.long)

    entropy_sum = 0.0
    top_prob_sum = 0.0
    nearest_dist_sum = 0.0
    sampled_dist_sum = 0.0

    for j, u in enumerate(private_nodes_tensor.tolist()):
        y_u = int(labels[u].item())
        candidate_idx = class_idx_tensors[y_u] if label_conditioning else all_idx

        candidate_feats = dict_features.index_select(0, candidate_idx)
        q_u = Q[u]

        distances = torch.linalg.norm(candidate_feats - q_u.unsqueeze(0), dim=1)
        scores = -distances
        logits = (epsilon_per_node / (2.0 * utility_sensitivity)) * scores
        probs = torch.softmax(logits, dim=0)

        entropy = -(probs * torch.log(probs.clamp_min(1e-12))).sum().item()
        top_prob = float(probs.max().item())

        local_idx = gumbel_max_sample(logits)
        proto_idx = int(candidate_idx[local_idx].item())

        nearest_dist_sum += float(distances.min().item())
        sampled_dist_sum += float(distances[local_idx].item())
        entropy_sum += entropy
        top_prob_sum += top_prob

        selected_proto_indices[j] = proto_idx
        selection_counts[proto_idx] += 1

        if (j + 1) % 5000 == 0:
            print(f'  sampled {j + 1}/{N} private nodes')

    used_ratio = float((selection_counts > 0).sum().item()) / float(max(K_global, 1))
    diagnostics = {
        'epsilon_total': float(epsilon_total),
        'epsilon_per_node': float(epsilon_per_node),
        'dict_size': int(K_global),
        'mean_entropy': float(entropy_sum / max(N, 1)),
        'mean_top_probability': float(top_prob_sum / max(N, 1)),
        'mean_nearest_distance': float(nearest_dist_sum / max(N, 1)),
        'mean_sampled_distance': float(sampled_dist_sum / max(N, 1)),
        'unique_selected_ratio': used_ratio,
    }

    assignments = {'proto_indices': selected_proto_indices, 'labels': private_labels}
    return assignments, diagnostics


class PrototypeAssignmentDataset(torch.utils.data.Dataset):
    """Memory-efficient dataset: stores indices into the shared public_dict."""
    def __init__(self, public_dict, proto_indices, labels):
        self.public_dict = public_dict
        self.proto_indices = proto_indices.long().cpu()
        self.labels = labels.long().cpu()

    def __len__(self):
        return self.proto_indices.numel()

    def __getitem__(self, idx):
        proto = self.public_dict[int(self.proto_indices[idx].item())]
        return Data(x=proto['x'], edge_index=proto['edge_index'], y=self.labels[idx].view(1))


class StandardGCN(nn.Module):
    def __init__(self, hidden_channels, num_features, num_classes):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)
        return self.lin(x)


def build_validation_loader(
    x_l2, labels, val_nodes, val_edge_index, query_hops=1, max_val_graphs=None, seed=0, batch_size=128,
):
    if max_val_graphs is not None:
        val_nodes = cap_indices(val_nodes, max_count=max_val_graphs, seed=seed)

    val_u = to_undirected(val_edge_index, num_nodes=x_l2.size(0))
    val_u = coalesce(val_u, num_nodes=x_l2.size(0))

    data_list = []
    for node in val_nodes.tolist():
        subset, sub_ei, _, _ = k_hop_subgraph(
            node, num_hops=query_hops, edge_index=val_u, relabel_nodes=True, num_nodes=x_l2.size(0),
        )
        data_list.append(Data(x=x_l2[subset], edge_index=sub_ei, y=labels[node].unsqueeze(0)))

    return DataLoader(data_list, batch_size=batch_size, shuffle=False), val_nodes


def train_gnn_from_assignments(
    assignments, public_dict, val_loader, num_features, num_classes,
    hidden_channels=16, epochs=50, batch_size=32, lr=0.01,
    feature_jitter_std=0.05, edge_dropout_p=0.10, verbose_every=10,
):
    """Train a GCN on synthetic prototype data; returns (model, best_val_acc)."""
    ds = PrototypeAssignmentDataset(public_dict, assignments['proto_indices'], assignments['labels'])
    use_cuda = device.type == 'cuda'
    loader_kwargs = {'batch_size': batch_size, 'shuffle': True,
                     'num_workers': 2 if use_cuda else 0, 'pin_memory': use_cuda}
    if loader_kwargs['num_workers'] > 0:
        loader_kwargs['persistent_workers'] = True
    loader = DataLoader(ds, **loader_kwargs)

    model = StandardGCN(hidden_channels, num_features, num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_val = 0.0
    best_state = None
    for epoch in range(1, epochs + 1):
        model.train()
        for batch in loader:
            batch = batch.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            x_aug = batch.x + torch.randn_like(batch.x) * feature_jitter_std
            ei_aug, _ = dropout_edge(batch.edge_index, p=edge_dropout_p)
            if ei_aug.numel() == 0:
                ei_aug = batch.edge_index
            out = model(x_aug, ei_aug, batch.batch)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()

        model.eval()
        correct = 0
        with torch.no_grad():
            for vb in val_loader:
                vb = vb.to(device, non_blocking=True)
                pred = model(vb.x, vb.edge_index, vb.batch).argmax(dim=1)
                correct += int((pred == vb.y).sum())
        val_acc = correct / max(len(val_loader.dataset), 1)
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if verbose_every > 0 and (epoch == 1 or epoch % verbose_every == 0):
            print(f'  Epoch {epoch:03d} | Val Acc: {val_acc:.4f} | Best: {best_val:.4f}')

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model, best_val


In [ ]:
# ==========================================
# CELL 4: RUN MM-edgeDP (epsilon=1.0 and epsilon=10000 sanity check)
# ==========================================
EPSILON_LIST_MAIN = [0.5, 1.0, 2.0, 4.0]  # used by sweeps below
EPSILON_PRIVATE   = 1.0
EPSILON_SANITY    = 10000.0

QUERY_MODE = 'one_hop'        # 'one_hop' | 'two_hop_concat'
LABEL_CONDITIONING = True
MIN_CLASS_FRACTION = 1.0
UTILITY_SENSITIVITY = 2.0 if QUERY_MODE == 'two_hop_concat' else 1.0

# Max prototype nodes is a scale guard: needed on ogbn-arxiv/PubMed, not on Cora/CiteSeer.
MAX_PROTO_NODES = 64 if DATASET_NAME in ('ogbn-arxiv', 'pubmed') else None

# -------- Build dictionary + validation loader --------
public_dict, public_proto_feats, class_to_proto_indices = build_class_conditional_public_dictionary(
    data_obj=data,
    x_features=x_l2,
    labels_tensor=data.y,
    public_nodes_tensor=public_nodes,
    pub_edges=pub_edge_index,
    num_classes=num_classes,
    dict_per_class=PRESET['dict_per_class'],
    num_hops=PRESET['walk_hops'],
    query_mode=QUERY_MODE,
    min_class_fraction=MIN_CLASS_FRACTION,
    max_proto_nodes=MAX_PROTO_NODES,
)

val_loader, val_nodes_used = build_validation_loader(
    x_l2=x_l2, labels=data.y, val_nodes=val_nodes, val_edge_index=val_edge_index,
    query_hops=PRESET['query_hops'], max_val_graphs=PRESET['max_val_nodes'], seed=SEED + 3,
    batch_size=PRESET['batch_size'] * 4,
)
print(f'Validation graphs used: {len(val_loader.dataset)}')

# Validation majority-class baseline
val_labels = data.y[val_nodes_used]
val_counts = torch.bincount(val_labels, minlength=num_classes)
majority_label = int(torch.argmax(val_counts).item())
majority_acc = float((val_labels == majority_label).float().mean().item())
print(f'Validation majority-class baseline acc: {majority_acc:.4f} (class={majority_label})')


def run_mm_edgedp(epsilon, label=''):
    print(f'\n=== MM-edgeDP: {label} (epsilon={epsilon}) ===')
    assignments, diag = synthesize_edge_dp_assignments(
        private_edges=private_edge_index,
        x_features=x_l2,
        labels=data.y,
        private_nodes_tensor=train_nodes,
        dict_features=public_proto_feats,
        class_to_proto_indices=class_to_proto_indices,
        epsilon_total=epsilon,
        utility_sensitivity=UTILITY_SENSITIVITY,
        query_mode=QUERY_MODE,
        label_conditioning=LABEL_CONDITIONING,
    )
    print(f"  diag: entropy={diag['mean_entropy']:.4f}  top_p={diag['mean_top_probability']:.4f}  "
          f"unique_ratio={diag['unique_selected_ratio']:.2%}  dict_size={diag['dict_size']}")
    model, best_val = train_gnn_from_assignments(
        assignments=assignments,
        public_dict=public_dict,
        val_loader=val_loader,
        num_features=data.x.size(1),
        num_classes=num_classes,
        hidden_channels=PRESET['hidden_channels'],
        epochs=PRESET['epochs'],
        batch_size=PRESET['batch_size'],
        feature_jitter_std=0.05,
        edge_dropout_p=0.10,
        verbose_every=max(1, PRESET['epochs'] // 5),
    )
    return model, best_val, diag


dp_model,        best_val_private, diag_private = run_mm_edgedp(EPSILON_PRIVATE, label='private')
dp_model_sanity, best_val_sanity,  diag_sanity  = run_mm_edgedp(EPSILON_SANITY,  label='sanity')

print('\n=== MM-edgeDP summary ===')
print(f'Best Val Acc | epsilon={EPSILON_PRIVATE}: {best_val_private:.4f}')
print(f'Best Val Acc | epsilon={EPSILON_SANITY}: {best_val_sanity:.4f}')
gap = best_val_sanity - best_val_private
print(f'Sanity gap (epsilon_high - epsilon_low): {gap:+.4f}')
if abs(gap) <= 0.02:
    print('Interpretation: dictionary/retrieval pipeline is the bottleneck (DP noise not dominant).')
elif gap > 0.02:
    print('Interpretation: DP noise contributes materially to the gap.')
else:
    print('Interpretation: high-epsilon run underperformed low-epsilon; re-run across seeds.')


## 4. Baseline 1 — Gaussian SGC (continuous Edge-DP)

Sum-aggregates private 1-hop neighbourhoods, then adds Gaussian noise calibrated to
(ε, δ)-Edge-DP. Trains an MLP on the noisy aggregate.


In [ ]:
# ==========================================
# CELL 5: BASELINE 1 — EDGE-DP GAUSSIAN SGC
# ==========================================

def run_edge_dp_sgc_baseline(
    data_obj, private_edges, val_edges, train_nodes_idx, val_nodes_idx,
    epsilon=1.0, delta=1e-5, epochs=50, hidden=16, lr=0.01,
):
    num_nodes    = data_obj.num_nodes
    num_features = data_obj.x.size(1)

    x_norm = F.normalize(data_obj.x, p=2, dim=1)

    # Private 1-hop sum aggregation
    priv_u = to_undirected(private_edges, num_nodes=num_nodes)
    priv_u = coalesce(priv_u, num_nodes=num_nodes)
    x_agg_train = torch.zeros_like(x_norm)
    if priv_u.numel() > 0:
        row, col = priv_u
        x_agg_train.scatter_add_(0, row.unsqueeze(1).expand(-1, num_features), x_norm[col])

    delta_l2 = 2.0
    sigma = (delta_l2 / epsilon) * torch.sqrt(torch.tensor(2.0 * torch.log(torch.tensor(1.25 / delta))))
    print(f'Gaussian SGC | epsilon={epsilon}, delta={delta}, sigma={sigma.item():.4f}')
    x_noisy_train = x_agg_train + torch.randn_like(x_agg_train) * sigma

    # Clean validation aggregation on val edges
    val_u = to_undirected(val_edges, num_nodes=num_nodes)
    val_u = coalesce(val_u, num_nodes=num_nodes)
    x_agg_val = torch.zeros_like(x_norm)
    if val_u.numel() > 0:
        vrow, vcol = val_u
        x_agg_val.scatter_add_(0, vrow.unsqueeze(1).expand(-1, num_features), x_norm[vcol])

    class DPMLP(nn.Module):
        def __init__(self, in_ch, h, out_ch):
            super().__init__()
            self.lin1 = nn.Linear(in_ch, h)
            self.lin2 = nn.Linear(h, out_ch)
        def forward(self, x):
            return self.lin2(F.relu(self.lin1(x)))

    model = DPMLP(num_features, hidden, num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_x = x_noisy_train[train_nodes_idx].to(device)
    train_y = data_obj.y[train_nodes_idx].to(device)
    val_x   = x_agg_val[val_nodes_idx].to(device)
    val_y   = data_obj.y[val_nodes_idx].to(device)

    best_val = 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(train_x)
        loss = criterion(out, train_y)
        loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            vp = model(val_x).argmax(dim=1)
            vacc = int((vp == val_y).sum()) / max(len(val_y), 1)
        best_val = max(best_val, vacc)
    print(f'Gaussian SGC best val acc: {best_val:.4f}')
    return model, best_val


sgc_results = {}
for eps in EPSILON_LIST_MAIN:
    _, acc = run_edge_dp_sgc_baseline(
        data_obj=data, private_edges=private_edge_index, val_edges=val_edge_index,
        train_nodes_idx=train_nodes, val_nodes_idx=val_nodes_used,
        epsilon=eps, delta=1e-5, epochs=PRESET['epochs'], hidden=PRESET['hidden_channels'],
    )
    sgc_results[eps] = acc

print('\n=== Gaussian SGC Summary ===')
for eps, acc in sgc_results.items():
    print(f'  epsilon={eps:.1f} -> val acc {acc:.4f}')


## 5. Baseline 2 — Edge Randomized Response (adjacency perturbation)

Keeps each true edge with probability `e^ε / (1 + e^ε)`; fills the dropped-edge budget
with random non-edges. A 2-layer GCN is then trained on the perturbed graph.


In [ ]:
# ==========================================
# CELL 6: BASELINE 2 — EDGE RANDOMIZED RESPONSE
# ==========================================

class GCN2L(nn.Module):
    def __init__(self, in_ch, hid, out_ch, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hid)
        self.conv2 = GCNConv(hid, out_ch)
        self.dropout = dropout
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)


def train_gcn_transductive(x_feat, edge_index, y_labels, train_idx, val_idx,
                           epochs=200, lr=0.01, hidden=256, weight_decay=5e-4):
    x_feat     = x_feat.to(device)
    edge_index = edge_index.to(device)
    y_labels   = y_labels.squeeze().to(device)
    model = GCN2L(x_feat.size(1), hidden, num_classes).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        out = model(x_feat, edge_index)
        loss = F.cross_entropy(out[train_idx], y_labels[train_idx])
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        preds = model(x_feat, edge_index).argmax(dim=1)
        val_acc = (preds[val_idx] == y_labels[val_idx]).float().mean().item()
    return model, val_acc


def apply_edge_rr(edge_index, num_nodes, epsilon):
    p_keep = math.exp(epsilon) / (1.0 + math.exp(epsilon))
    ei = coalesce(to_undirected(edge_index, num_nodes=num_nodes), num_nodes=num_nodes)
    m = ei.size(1)
    keep = torch.rand(m) < p_keep
    kept = ei[:, keep]
    n_drop = int((~keep).sum())
    if n_drop > 0:
        fsrc = torch.randint(0, num_nodes, (n_drop,))
        fdst = torch.randint(0, num_nodes, (n_drop,))
        valid = fsrc != fdst
        fake = torch.stack([fsrc[valid], fdst[valid]])
        ei_rr = torch.cat([kept, fake], dim=1)
    else:
        ei_rr = kept
    ei_rr = coalesce(to_undirected(ei_rr, num_nodes=num_nodes), num_nodes=num_nodes)
    return ei_rr, p_keep, n_drop


rr_results = {}
# Hidden size for Edge RR baseline: keep the original 256 for small graphs, scale down for ogbn-arxiv.
RR_HIDDEN = 256 if DATASET_NAME != 'ogbn-arxiv' else 128
RR_EPOCHS = 200 if DATASET_NAME != 'ogbn-arxiv' else 100

for eps in EPSILON_LIST_MAIN:
    ei_rr, p_keep, n_drop = apply_edge_rr(data.edge_index, data.num_nodes, eps)
    print(f'Edge RR | epsilon={eps} | p_keep={p_keep:.3f} | dropped={n_drop} | new edges={ei_rr.size(1)}')
    _, val_acc = train_gcn_transductive(
        x_l2, ei_rr, data.y, train_nodes, val_nodes_used,
        epochs=RR_EPOCHS, hidden=RR_HIDDEN,
    )
    rr_results[eps] = val_acc
    print(f'  val acc = {val_acc:.4f}')

print('\n=== Edge RR Summary ===')
for eps, acc in rr_results.items():
    print(f'  epsilon={eps:.1f} -> {acc:.4f}')


## 6. Baseline 3 — GAP-EDP (Aggregation Perturbation)

Sajadmanesh & Gatica-Perez (USENIX Security 2023), arXiv:2203.00949.

Note: this is a Node-DP method, not Edge-DP. Included as a third point of comparison
against MM-edgeDP and Edge RR; the guarantee differs.


In [ ]:
# ==========================================
# CELL 7: BASELINE 3 — GAP-EDP (AGGREGATION PERTURBATION)
# ==========================================

K_HOPS     = 2
ENC_DIM    = 16
ENC_EPOCHS = 50
CLF_EPOCHS = 100
DELTA      = 1e-5


class EncoderMLP(nn.Module):
    def __init__(self, in_ch, enc_dim, out_ch):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(in_ch, enc_dim * 2), nn.ReLU(),
            nn.Linear(enc_dim * 2, enc_dim),
        )
        self.head = nn.Linear(enc_dim, out_ch)
    def forward(self, x):
        z = self.enc(x)
        return self.head(z), z


def calibrate_sigma(epsilon, delta, K):
    b = math.sqrt(2 * K * math.log(1.25 / delta))
    t = (-b + math.sqrt(b ** 2 + 2 * K * epsilon)) / K
    return 1.0 / t


def private_multi_hop_aggregation(x_init, edge_index, n_nodes, sigma, K):
    ei = coalesce(to_undirected(edge_index, num_nodes=n_nodes), num_nodes=n_nodes)
    row, col = ei
    d = x_init.size(1)
    x_hat = x_init.clone()
    hops = []
    for _ in range(K):
        x_agg = torch.zeros(n_nodes, d)
        x_agg.scatter_add_(0, row.unsqueeze(1).expand(-1, d), x_hat[col])
        x_tilde = x_agg + torch.randn_like(x_agg) * sigma
        x_hat = F.normalize(x_tilde, p=2, dim=1)
        hops.append(x_hat.clone())
    return hops


class ClassifierMLP(nn.Module):
    def __init__(self, in_ch, hidden, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_ch, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_ch),
        )
    def forward(self, x):
        return self.net(x)


def train_gap_classifier(x_feat, y_labels, train_idx, val_idx, epochs, hidden=64):
    x_feat   = x_feat.to(device)
    y_labels = y_labels.squeeze().to(device)
    model = ClassifierMLP(x_feat.size(1), hidden=hidden, out_ch=num_classes).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        F.cross_entropy(model(x_feat[train_idx]), y_labels[train_idx]).backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        preds = model(x_feat).argmax(dim=1).cpu()
        val_acc = (preds[val_idx] == y_labels.cpu()[val_idx]).float().mean().item()
    return model, val_acc


# -------- Step 1: pre-train encoder on node features only --------
print('\n[GAP-EDP 1/3] Pre-training encoder on node features only...')
enc_model = EncoderMLP(x_l2.size(1), ENC_DIM, num_classes).to(device)
enc_opt = torch.optim.Adam(enc_model.parameters(), lr=1e-3)
x_gpu = x_l2.to(device)
y_gpu = data.y.squeeze().to(device)
tn_gpu = train_nodes.to(device)

for _ in range(ENC_EPOCHS):
    enc_model.train()
    enc_opt.zero_grad()
    logits, _ = enc_model(x_gpu[tn_gpu])
    F.cross_entropy(logits, y_gpu[tn_gpu]).backward()
    enc_opt.step()

enc_model.eval()
with torch.no_grad():
    _, x_enc = enc_model(x_gpu)
    x_enc = x_enc.cpu()
x0 = F.normalize(x_enc, p=2, dim=1)
print(f'  Encoded shape: {x0.shape}  max-norm={x0.norm(dim=1).max():.4f}')


# -------- Step 2 & 3: PMA + classification sweep --------
print('\n[GAP-EDP 2/3] Running PMA + classification sweep...')
gap_results = {}
for eps in EPSILON_LIST_MAIN:
    sigma = calibrate_sigma(eps, DELTA, K_HOPS)
    hops = private_multi_hop_aggregation(x0, data.edge_index, data.num_nodes, sigma, K_HOPS)
    x_combined = torch.cat([x0] + hops, dim=1)
    _, val_acc = train_gap_classifier(
        x_combined, data.y, train_nodes, val_nodes_used, CLF_EPOCHS,
        hidden=64 if DATASET_NAME != 'ogbn-arxiv' else 128,
    )
    gap_results[eps] = val_acc
    print(f'  epsilon={eps} sigma={sigma:.4f} -> val acc {val_acc:.4f}')

print('\n[GAP-EDP 3/3] Summary')
for eps, acc in gap_results.items():
    print(f'  epsilon={eps:.1f} -> {acc:.4f}')


# -------- Comparison table --------
print('\n' + '=' * 70)
print('BASELINE COMPARISON on', DATASET_NAME)
print(f"{'epsilon':<10}{'Gaussian SGC':>14}{'Edge RR':>12}{'GAP-EDP':>12}{'MM-edgeDP':>14}")
print('-' * 62)
for eps in EPSILON_LIST_MAIN:
    sgc = sgc_results.get(eps, float('nan'))
    rr  = rr_results.get(eps, float('nan'))
    gap = gap_results.get(eps, float('nan'))
    mm  = best_val_private if abs(eps - EPSILON_PRIVATE) < 1e-9 else float('nan')
    print(f'{eps:<10.1f}{sgc:>14.4f}{rr:>12.4f}{gap:>12.4f}{mm:>14.4f}')
print('(MM-edgeDP only run at eps=1.0 in the main cell; use a sweep cell to fill other eps.)')


## 7. Edge-inference attacks (Threat Models A and B)

- **Threat Model A (black-box)**: adversary queries the trained GCN with a single node's
  features (empty edge index) and reads the softmax output.
- **Threat Model B (white-box)**: adversary runs the same empty-edge query and reads the
  penultimate-layer embedding.

For each pair of private nodes, the adversary predicts whether a private edge exists.
AUC = 0.5 is no leakage; AUC = 1.0 is full exposure.


In [ ]:
# ==========================================
# CELL 8: EDGE-INFERENCE ATTACKS (Threat Models A and B)
# ==========================================

@torch.no_grad()
def get_embeddings_whitebox(model, nodes, x_features):
    """Threat Model B: penultimate-layer embedding, no private edges."""
    model.eval()
    empty_ei = torch.zeros(2, 0, dtype=torch.long, device=device)
    batch    = torch.zeros(1, dtype=torch.long, device=device)
    out = {}
    for node in nodes:
        x_node = x_features[int(node)].unsqueeze(0).to(device)
        h = model.conv1(x_node, empty_ei).relu()
        h = model.conv2(h, empty_ei).relu()
        out[int(node)] = global_mean_pool(h, batch).squeeze(0).cpu()
    return out


@torch.no_grad()
def get_predictions_blackbox(model, nodes, x_features):
    """Threat Model A: softmax outputs, no private edges."""
    model.eval()
    empty_ei = torch.zeros(2, 0, dtype=torch.long, device=device)
    batch    = torch.zeros(1, dtype=torch.long, device=device)
    out = {}
    for node in nodes:
        x_node = x_features[int(node)].unsqueeze(0).to(device)
        logits = model(x_node, empty_ei, batch)
        out[int(node)] = torch.softmax(logits, dim=-1).squeeze(0).cpu()
    return out


def build_attack_xy(rep_dict, positive_ei, private_nodes_set, mode='whitebox', seed=42):
    """Balanced (X, y): positive = real private edges, negative = random non-edges."""
    rng = np.random.default_rng(seed)
    private_arr = np.array(list(private_nodes_set))

    pos_set = set()
    for u, v in zip(positive_ei[0].tolist(), positive_ei[1].tolist()):
        if u in rep_dict and v in rep_dict and u != v:
            pos_set.add((min(u, v), max(u, v)))
    pos_pairs = list(pos_set)

    if len(pos_pairs) == 0:
        raise RuntimeError('No positive edges overlap with the private node representation set.')

    neg_pairs, neg_keys = [], set()
    max_tries = 50 * len(pos_pairs)
    tries = 0
    while len(neg_pairs) < len(pos_pairs) and tries < max_tries:
        tries += 1
        u, v = rng.choice(private_arr, 2, replace=False).tolist()
        u, v = int(u), int(v)
        key = (min(u, v), max(u, v))
        if key not in pos_set and key not in neg_keys and u in rep_dict and v in rep_dict:
            neg_pairs.append((u, v))
            neg_keys.add(key)

    X, y = [], []
    for label, pairs in [(1, pos_pairs), (0, neg_pairs)]:
        for u, v in pairs:
            a, b = rep_dict[u].numpy(), rep_dict[v].numpy()
            feat = (a * b) if mode == 'whitebox' else np.concatenate([np.abs(a - b), a * b])
            X.append(feat)
            y.append(label)
    return np.array(X), np.array(y)


def run_attack(rep_dict, positive_ei, private_nodes_set, mode, label):
    X, y = build_attack_xy(rep_dict, positive_ei, private_nodes_set, mode=mode)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=1000, solver='lbfgs')
    clf.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
    n_pos = int((y == 1).sum())
    print(f'  [{label}] n_pos={n_pos}  AUC={auc:.4f}')
    return auc


# -------- Run attacks on the two MM-edgeDP models trained earlier --------
private_nodes_set = set(train_nodes.tolist())
attack_results = {}

for eps_label, model in [(EPSILON_PRIVATE, dp_model), (EPSILON_SANITY, dp_model_sanity)]:
    print(f'\n===== Edge-inference attack on MM-edgeDP | epsilon={eps_label} =====')
    wb_reps = get_embeddings_whitebox(model, train_nodes.tolist(), x_l2)
    auc_wb  = run_attack(wb_reps, private_edge_index, private_nodes_set,
                         mode='whitebox', label=f'whitebox eps={eps_label}')
    bb_reps = get_predictions_blackbox(model, train_nodes.tolist(), x_l2)
    auc_bb  = run_attack(bb_reps, private_edge_index, private_nodes_set,
                         mode='blackbox', label=f'blackbox eps={eps_label}')
    attack_results[eps_label] = {'whitebox': auc_wb, 'blackbox': auc_bb}


# -------- Summary --------
print('\n' + '=' * 55)
print(f'Edge-inference attack summary on {DATASET_NAME} (MM-edgeDP):')
print(f"{'':30s} {'eps=' + str(EPSILON_PRIVATE):>12s} {'eps=' + str(EPSILON_SANITY):>12s}")
print(f"{'White-box AUC (Threat B)':30s} "
      f"{attack_results[EPSILON_PRIVATE]['whitebox']:>12.4f} "
      f"{attack_results[EPSILON_SANITY]['whitebox']:>12.4f}")
print(f"{'Black-box AUC (Threat A)':30s} "
      f"{attack_results[EPSILON_PRIVATE]['blackbox']:>12.4f} "
      f"{attack_results[EPSILON_SANITY]['blackbox']:>12.4f}")
print('\nAUC=0.5 -> no leakage; AUC=1.0 -> full exposure.')
print('Expectation: epsilon=1.0 AUC should be closer to 0.5 than epsilon=10000.')


## 8. Final summary

All numbers for the current dataset selection, with MM-edgeDP reported at ε=1.0.
To sweep MM-edgeDP across the same ε list as the baselines, wrap the `run_mm_edgedp`
call from Section 3 in a loop over `EPSILON_LIST_MAIN`.


In [ ]:
# ==========================================
# CELL 9: FINAL SUMMARY TABLE
# ==========================================
print(f'Dataset: {DATASET_NAME}')
print(f'Nodes: {data.num_nodes} | Edges: {data.edge_index.size(1)} | Classes: {num_classes} | Feature dim: {data.x.size(1)}')
print(f'Split -> Public: {len(public_nodes)} | Private train: {len(train_nodes)} | Val: {len(val_nodes_used)}')
print()
print('--- Accuracy across baselines ---')
print(f"{'epsilon':<10}{'Gaussian SGC':>14}{'Edge RR':>12}{'GAP-EDP':>12}{'MM-edgeDP':>14}")
for eps in EPSILON_LIST_MAIN:
    sgc = sgc_results.get(eps, float('nan'))
    rr  = rr_results.get(eps, float('nan'))
    gap = gap_results.get(eps, float('nan'))
    mm  = best_val_private if abs(eps - EPSILON_PRIVATE) < 1e-9 else float('nan')
    print(f'{eps:<10.1f}{sgc:>14.4f}{rr:>12.4f}{gap:>12.4f}{mm:>14.4f}')

print(f"\nMM-edgeDP at epsilon={EPSILON_SANITY} (sanity): {best_val_sanity:.4f}")

print('\n--- Edge-inference attack AUC (MM-edgeDP) ---')
print(f"{'epsilon':<10}{'White-box (B)':>16}{'Black-box (A)':>16}")
for eps in [EPSILON_PRIVATE, EPSILON_SANITY]:
    print(f"{eps:<10}{attack_results[eps]['whitebox']:>16.4f}{attack_results[eps]['blackbox']:>16.4f}")
